# Import Modules

In [28]:
#high level modules
import os
import importlib.util
from functools import partial
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [29]:

def import_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

universal_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/"
this_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/forecast_rollout/"

import_from_path("universals", os.path.join(universal_dir, "universal_functions.py"))
from universals import load_pickle_file

import_from_path("forecast_fx", os.path.join(this_dir, "forecast_functions.py"))
from forecast_fx import prep_features, static_regime, pulsed_regime, rollout_forecast

dump_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/test_forecast/"


# Import models

In [30]:
model_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/dump/five_ten/"

model_1 = load_pickle_file("model_1.pkl", model_dir)
model_2 = load_pickle_file("model_2.pkl", model_dir)
model_3 = load_pickle_file("model_3.pkl", model_dir)
model_4 = load_pickle_file("model_4.pkl", model_dir)

# Import data

In [31]:
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/data/"

test = pd.read_csv(os.path.join(data_dir, "test.csv"))

test["date"] = pd.to_datetime(test["date"])

# filter test set for data from 2015, then from 2022
test_2015 = test[test["date"].dt.year == 2015].copy()
test_2022 = test[test["date"].dt.year == 2022].copy()

print(test_2022)

          date  mean_1m_temp_degC  mean_0_5m_temp_degC  mean_1m_temp_degC_m1  \
96  2022-05-27          -1.499162            -1.477215             -1.573920   
97  2022-05-28          -1.493109            -1.117342             -1.499162   
98  2022-05-29          -1.691044            -1.216932             -1.493109   
99  2022-05-30          -1.719930            -1.287667             -1.691044   
100 2022-05-31          -1.828276            -1.607793             -1.719930   
..         ...                ...                  ...                   ...   
220 2022-09-28          -0.533804            -0.087108             -0.551739   
221 2022-09-29          -0.576934            -0.055805             -0.533804   
222 2022-09-30          -0.701880            -0.109209             -0.576934   
223 2022-10-01          -0.794750            -0.232793             -0.701880   
224 2022-10-02          -0.871791            -0.305744             -0.794750   

     mean_0_5m_temp_degC_m1  total_sola

# Roll out forecast

Here, we're just iteratively modeling, for each of the 4 models for the next 7 days. First, we need to remove any columns that aren't in the model proper, and we need to make sure the order of the columns is the same as the training set. To do this, we'll just select the feature names in the proper order.

First thing is that we need to prep the data for forecasting. This step uses a single date and grabs the data seven days in the future. Data are recoded as needed, and a forecast date column is created - this is the date that the forecast is made. 

In [32]:

all_dates_2015 = pd.date_range(start="2015-07-01", end="2015-10-08", freq="D").strftime("%Y-%m-%d")

control_dataset = pd.concat([prep_features(one_date=d, data=test_2015, regime="control") for d in all_dates_2015])
control_dataset["forecast_date"] = pd.to_datetime(control_dataset["forecast_date"])


In [33]:
control_dataset.columns

Index(['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'mean_1m_temp_degC_m1',
       'mean_0_5m_temp_degC_m1', 'total_solar_radiation',
       'total_solar_radiation_m1', 'total_solar_radiation_m2', 'mean_air_temp',
       'min_air_temp', 'max_air_temp', 'mean_air_temp_m1', 'min_air_temp_m1',
       'max_air_temp_m1', 'mean_rel_hum_m1', 'mean_rel_hum_m2', 'pump_cfs_m1',
       'pump_cfs_m2', 'pump_cfs_m3', 'mean_wind', 'max_wind', 'mean_wind_m1',
       'max_wind_m1', 'nf_cfs_m1', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m1', 'chipmunk_cfs_m2', 'chipmunk_cfs_m3',
       'chipmunk_cfs_m4', 'forecast_date'],
      dtype='object')

In [34]:
# get the feature names from the operational training file
feature_names = pd.read_csv(os.path.join(universal_dir, "data/training_1.csv")).columns

# remove date, and two target columns (first 3 columns)
feature_names = feature_names[3:]
feature_names = list(feature_names) + ["forecast_date"]

# and get those features in order for each dataset! (but we also need the forecast date!)
control_for_model = control_dataset[feature_names].copy()

unique_forecast = control_for_model["forecast_date"].unique()

And now rollout the forecast per unique valid date.

In [35]:
control_forecasts = (pd.concat([rollout_forecast(data = control_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_forecast]))

And now we need to transform the data back to temp. To do this we need the transformation file.

In [36]:
transformations = pd.read_csv("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/data/mu_sigma.csv", index_col=0)


In [37]:
# we need to back transform the data for mean 1m and mean 0.5m
control_forecasts["mean_1m_temp_degC"] = control_forecasts["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
control_forecasts["mean_0_5m_temp_degC"] = control_forecasts["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

export_features = ["forecast_date", "valid_date", "model", "mean_1m_temp_degC", "mean_0_5m_temp_degC"]
control_forecasts[export_features]
# save the files to csv
control_forecasts.to_csv(os.path.join(dump_dir, "forecasted_temp_2015_control_collated.csv"), index=False)

In [38]:

all_dates_2022 = pd.date_range(start="2022-05-27", end="2022-10-02", freq="D").strftime("%Y-%m-%d")

control_dataset = pd.concat([prep_features(one_date=d, data=test_2022, regime="control") for d in all_dates_2022])
control_dataset["forecast_date"] = pd.to_datetime(control_dataset["forecast_date"])


In [39]:
# and get those features in order for each dataset! (but we also need the forecast date!)
control_for_model = control_dataset[feature_names].copy()

unique_forecast = control_for_model["forecast_date"].unique()

In [40]:
control_forecasts = (pd.concat([rollout_forecast(data = control_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_forecast]))

In [41]:
# we need to back transform the data for mean 1m and mean 0.5m
control_forecasts["mean_1m_temp_degC"] = control_forecasts["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
control_forecasts["mean_0_5m_temp_degC"] = control_forecasts["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

export_features = ["forecast_date", "valid_date", "model", "mean_1m_temp_degC", "mean_0_5m_temp_degC"]
control_forecasts[export_features]
# save the files to csv
control_forecasts.to_csv(os.path.join(dump_dir, "forecasted_temp_2022_control_collated.csv"), index=False)